# Course Bootstrap
## 初始化課程專案到 Google Drive

1. 執行環境偵測與 Drive 掛載。
2. 設定同步模式：
   - `fresh_install`: 重新安裝整份課程 repo。
   - `update_preserve_results`: 更新最新教材，但保留 `assets/results/` 產出。
3. 從 GitHub 下載最新課程 snapshot，同步到 Google Drive。
4. 檢查課程資料夾內容是否完整，並補齊 Colab 套件。

> 注意：更新模式會覆蓋 repo 內的 notebook 與 `src/` 程式碼；如果你直接改教材內容，請先另存副本。


In [ ]:
from pathlib import Path

try:
    from google.colab import drive  # type: ignore[import-not-found]

    IN_COLAB = True
except ModuleNotFoundError:
    drive = None
    IN_COLAB = False

COURSE_TITLE = "Python 籃球運動資料分析課程"

if IN_COLAB:
    drive.mount("/content/drive")
    ROOT = Path("/content/drive/MyDrive/basketball_hackathon")
    TARGET_DIR = ROOT / "course"
else:
    ROOT = Path.cwd().resolve()
    TARGET_DIR = ROOT
    print("本機驗證模式：使用目前的課程 repo。")

ROOT.mkdir(parents=True, exist_ok=True)

print(COURSE_TITLE)
print("Drive 工作目錄:", ROOT)
print("課程資料夾:", TARGET_DIR)


In [ ]:
import shutil
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/henry753951/basketball-hackathon-course.git"
REPO_BRANCH = "main"
SYNC_MODE = "update_preserve_results"  # 或改成 "fresh_install"
PRESERVE_PATHS = ("assets/results",)

if IN_COLAB:
    LOCAL_CLONE_DIR = Path("/content/basketball_hackathon_course_tmp")
    if LOCAL_CLONE_DIR.exists():
        shutil.rmtree(LOCAL_CLONE_DIR)

    cmd = [
        "git",
        "clone",
        "--depth",
        "1",
        "-b",
        REPO_BRANCH,
        REPO_URL,
        str(LOCAL_CLONE_DIR),
    ]
    print("$", " ".join(cmd))
    subprocess.run(cmd, check=True)
else:
    LOCAL_CLONE_DIR = TARGET_DIR
    print("本機驗證模式：略過 git clone。")

if str(LOCAL_CLONE_DIR) not in sys.path:
    sys.path.insert(0, str(LOCAL_CLONE_DIR))

from src.course_setup import replace_tree_from_snapshot, rsync_tree  # noqa: E402

print("來源資料夾:", LOCAL_CLONE_DIR)
print("同步模式:", SYNC_MODE)
if SYNC_MODE == "update_preserve_results":
    print("保留路徑:", ", ".join(PRESERVE_PATHS))
print("注意：更新模式只保留 assets/results，repo 內 notebook / src 仍會更新。")


In [ ]:
import shutil

if IN_COLAB:
    if SYNC_MODE == "fresh_install":
        if TARGET_DIR.exists():
            shutil.rmtree(TARGET_DIR)
        rsync_tree(LOCAL_CLONE_DIR, TARGET_DIR)
    elif SYNC_MODE == "update_preserve_results":
        replace_tree_from_snapshot(
            LOCAL_CLONE_DIR,
            TARGET_DIR,
            preserve_paths=PRESERVE_PATHS,
        )
    else:
        raise ValueError(f"不支援的 SYNC_MODE: {SYNC_MODE}")
else:
    print("本機驗證模式：課程檔案已在目前 repo 中。")

print("課程檔案已準備完成:", TARGET_DIR)


In [ ]:
import subprocess
import sys

requirements = TARGET_DIR / "requirements.txt"
if IN_COLAB and requirements.exists():
    print("正在確認 Colab 課程套件...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(requirements)], check=True)
elif IN_COLAB:
    print("找不到 requirements.txt，請檢查 repo 是否完整。")
else:
    print("本機驗證模式：套件由 uv sync 管理，略過 pip install。")

print("\n課程資料夾內容：")
for p in sorted(TARGET_DIR.iterdir()):
    print("-", p.name)


In [ ]:
if IN_COLAB:
    # Google Drive 路徑:
    drive_path = Path("/content/drive/MyDrive/basketball_hackathon")
    if drive_path.exists():
        print(f"課程資料夾已建立於 {drive_path}")
        print("https://drive.google.com")
    